In [ ]:
"""Shared GROMACS setup for the p16-CDK6 simulations. Runs inside a Kaggle kernel."""
import os, subprocess, sys, shutil, textwrap, time
from pathlib import Path
import urllib.request

WORK = Path('/kaggle/working')
MM = Path('/kaggle/tmp/mm')
ENV = 'md'


def sh(cmd, inp=None, check=True, tail=25, cwd=None, quiet=False):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, input=inp, text=True, capture_output=True, cwd=cwd, executable='/bin/bash')
    out = r.stdout + r.stderr
    if not quiet or r.returncode:
        print(f'$ {cmd[:150]}  [{time.time() - t0:.0f}s, exit {r.returncode}]')
        print('\n'.join(out.splitlines()[-tail:]))
    if check and r.returncode:
        raise RuntimeError(f'failed: {cmd[:150]}')
    return out


def install():
    """Micromamba env with a CUDA build of GROMACS and the Python model-building tools."""
    MM.mkdir(parents=True, exist_ok=True)
    sh(f'cd {MM} && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba')
    print(sh(f'{MM}/bin/micromamba search -c conda-forge "gromacs>=2023" 2>&1 | tail -14', check=False, quiet=True))
    sh(f'{MM}/bin/micromamba create -y -q -r {MM}/r -n {ENV} -c conda-forge "gromacs=*=*cuda*" python=3.11 pdbfixer openmm numpy', check=False, tail=8)
    gmx = f'{MM}/r/envs/{ENV}/bin.AVX2_256/gmx'
    if not Path(gmx).exists():
        gmx = f'{MM}/r/envs/{ENV}/bin/gmx'
    return gmx


def gmx_ok(gmx):
    if not Path(gmx).exists():
        return False
    v = sh(f'{gmx} --version', check=False, quiet=True)
    print('\n'.join(l for l in v.splitlines() if any(k in l for k in ('GROMACS version', 'GPU support', 'SIMD', 'FFT library'))))
    return 'GPU support:        CUDA' in v or 'CUDA' in [l for l in v.splitlines() if 'GPU support' in l][0]


EM = """integrator=steep
emtol=1000
emstep=0.01
nsteps=50000
cutoff-scheme=Verlet
coulombtype=PME
rcoulomb=1.0
rvdw=1.0
pbc=xyz
"""
BASE = """cutoff-scheme=Verlet
coulombtype=PME
rcoulomb=1.0
vdwtype=cut-off
rvdw=1.0
DispCorr=EnerPres
pbc=xyz
tcoupl=V-rescale
tc-grps=System
tau_t=1.0
ref_t=310
constraints=h-bonds
constraint-algorithm=lincs
"""
NVT = BASE + """integrator=md
define=-DPOSRES
dt=0.002
nsteps=100000
nstenergy=5000
nstlog=5000
pcoupl=no
gen_vel=yes
gen_temp=310
gen_seed={seed}
"""
NPT = BASE + """integrator=md
define=-DPOSRES
refcoord_scaling=com
dt=0.002
nsteps=250000
nstenergy=5000
nstlog=5000
pcoupl=C-rescale
tau_p=2.0
ref_p=1.0
compressibility=4.5e-5
continuation=yes
gen_vel=no
"""
PROD = BASE + """integrator=md
dt=0.004
mass-repartition-factor=3
nsteps={nsteps}
nstxout-compressed=12500
compressed-x-precision=1000
nstenergy=12500
nstlog=25000
pcoupl=C-rescale
tau_p=5.0
ref_p=1.0
compressibility=4.5e-5
continuation=yes
gen_vel=no
"""


In [ ]:
"""Kernel H: build the four p16-CDK6 systems and measure real GROMACS speed on this GPU."""
import json, re
import numpy as np

print(sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2', quiet=True))
GMX = install()
print('gmx path', GMX, Path(GMX).exists())
ok = gmx_ok(GMX)
print('conda CUDA build usable:', ok)
if not ok:
    print('Building GROMACS from source with CUDA')
    sh(f'{MM}/bin/micromamba create -y -q -r {MM}/r -n {ENV} -c conda-forge python=3.11 pdbfixer openmm numpy cmake make "gcc=12" "g++=12" "cuda-nvcc" "cuda-version=12" "cudatoolkit=11.8" 2>&1 | tail -3', check=False)
    sh('cd /kaggle/tmp && curl -Ls https://ftp.gromacs.org/gromacs/gromacs-2024.4.tar.gz | tar xz')
    env = f'{MM}/r/envs/{ENV}'
    sh(f'cd /kaggle/tmp/gromacs-2024.4 && mkdir -p b && cd b && export PATH={env}/bin:$PATH && cmake .. -DGMX_GPU=CUDA -DGMX_BUILD_OWN_FFTW=ON -DGMX_SIMD=AVX2_256 -DCMAKE_INSTALL_PREFIX=/kaggle/tmp/gmx -DGMX_DEFAULT_SUFFIX=OFF 2>&1 | tail -5 && make -j$(nproc) install 2>&1 | tail -2', tail=8)
    GMX = '/kaggle/tmp/gmx/bin/gmx'
    print('source build usable:', gmx_ok(GMX))
PY = f'{MM}/r/envs/{ENV}/bin/python'

BUILD = r'''
import sys, urllib.request, numpy as np
from pdbfixer import PDBFixer
from openmm.app import PDBFile
name, muts = sys.argv[1], sys.argv[2:]
def get(p):
    return urllib.request.urlopen(f'https://files.rcsb.org/view/{p}.pdb').read().decode().splitlines()
def atoms(lines, ch):
    return [l for l in lines if l.startswith('ATOM') and l[21] == ch and l[16] in ' A']
def resi(l): return int(l[22:26])
def xyz(l): return np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
a, b = atoms(get('1BI7'), 'A'), atoms(get('1BI7'), 'B')
t = atoms(get('1BLX'), 'A')
ca = lambda ls, rs: {resi(l): xyz(l) for l in ls if l[12:16].strip() == 'CA' and resi(l) in rs}
fit = [r for r in list(range(36, 49)) + list(range(72, 86))]
P, Q = ca(t, fit), ca(a, fit)
keys = sorted(set(P) & set(Q))
X, Y = np.array([P[k] for k in keys]), np.array([Q[k] for k in keys])
cx, cy = X.mean(0), Y.mean(0)
U, S, Vt = np.linalg.svd((X - cx).T @ (Y - cy))
d = np.sign(np.linalg.det(Vt.T @ U.T))
R = Vt.T @ np.diag([1, 1, d]) @ U.T
rmsd = np.sqrt((((X - cx) @ R.T + cy - Y) ** 2).sum(1).mean())
print(f'graft fit on {len(keys)} CA: {rmsd:.2f} A')
graft = []
for l in t:
    if 49 <= resi(l) <= 71:
        v = (xyz(l) - cx) @ R.T + cy
        graft.append(l[:30] + f'{v[0]:8.3f}{v[1]:8.3f}{v[2]:8.3f}' + l[54:])
C48 = [xyz(l) for l in a if resi(l) == 48 and l[12:16].strip() == 'C'][0]
N49 = [xyz(l) for l in graft if resi(l) == 49 and l[12:16].strip() == 'N'][0]
C71 = [xyz(l) for l in graft if resi(l) == 71 and l[12:16].strip() == 'C'][0]
N72 = [xyz(l) for l in a if resi(l) == 72 and l[12:16].strip() == 'N'][0]
print(f'junctions: 48-49 {np.linalg.norm(C48 - N49):.2f} A, 71-72 {np.linalg.norm(C71 - N72):.2f} A')
Xg, Xb = np.array([xyz(l) for l in graft]), np.array([xyz(l) for l in b])
print(f'closest graft to p16 atom: {np.linalg.norm(Xg[:, None] - Xb[None], axis=2).min():.2f} A, graft atoms {len(graft)}')
merged = [l for l in a if resi(l) <= 48] + graft + [l for l in a if resi(l) >= 72]
open('in.pdb', 'w').write('\n'.join(merged + ['TER'] + b + ['TER', 'END']) + '\n')
fx = PDBFixer(filename='in.pdb')
by_chain = {}
for m in muts:
    c, code = m.split(':')
    by_chain.setdefault(c, []).append(code)
for c, codes in by_chain.items():
    fx.applyMutations(codes, c)
    print('mutated', c, codes)
chains = list(fx.topology.chains())
n = [len(list(c.residues())) for c in chains]
print('chains', [c.id for c in chains], n)
fx.findMissingResidues()
fx.missingResidues = {(0, 0): ['ACE'], (0, n[0]): ['NME'], (1, 0): ['ACE'], (1, n[1]): ['NME']}
fx.findNonstandardResidues(); fx.replaceNonstandardResidues()
fx.findMissingAtoms()
print('missing atoms in', len(fx.missingAtoms), 'residues; terminals', len(fx.missingTerminals))
fx.missingTerminals = {}
fx.addMissingAtoms(seed=2026)
PDBFile.writeFile(fx.topology, fx.positions, open('fixed.pdb', 'w'), keepIds=True)
out = []
for l in open('fixed.pdb'):
    if l.startswith('ATOM') or l.startswith('HETATM'):
        l = 'ATOM  ' + l[6:]
        if l[17:20] == 'NME' and l[12:16].strip() == 'C':
            l = l[:12] + ' CH3' + l[16:]
    out.append(l)
open('fixed.pdb', 'w').writelines(out)
seq = {}
for l in out:
    if l.startswith('ATOM') and l[12:16].strip() == 'CA': seq[(l[21], int(l[22:26]))] = l[17:20]
print('residues', len(seq), 'at mutated sites', {k: seq[k] for k in seq if k in [('B', 24), ('B', 84), ('B', 123)]})
'''
(WORK / 'build_model.py').write_text(BUILD)
for f, txt in [('em.mdp', EM), ('nvt.mdp', NVT.format(seed=1)), ('npt.mdp', NPT), ('md.mdp', PROD.format(nsteps=10000))]:
    (WORK / f).write_text(txt)

SYS = {'wt': [], 'r24p': ['B:ARG-24-PRO'], 'd84n': ['B:ASP-84-ASN'], 'h123q': ['B:HIS-123-GLN']}
for name, muts in SYS.items():
    d = WORK / 'systems' / name
    d.mkdir(parents=True, exist_ok=True)
    print('=' * 20, name)
    sh(f'{PY} {WORK}/build_model.py {name} {" ".join(muts)}', cwd=d)
    g = f'{GMX}'
    out = sh(f'{g} pdb2gmx -f fixed.pdb -o conf.gro -p topol.top -i posre.itp -ff amber99sb-ildn -water tip3p -ignh -chainsep id', cwd=d, check=False, tail=6)
    if 'Fatal error' in out or not (d / 'conf.gro').exists():
        print('default termini failed; selecting by hand')
        print(sh(f'{g} pdb2gmx -f fixed.pdb -o conf.gro -p topol.top -i posre.itp -ff amber99sb-ildn -water tip3p -ignh -chainsep id -ter', inp='', cwd=d, check=False, tail=40))
        raise SystemExit('stop: inspect termini')
    sh(f'{g} editconf -f conf.gro -o box.gro -c -d 1.0 -bt dodecahedron', cwd=d, tail=3)
    sh(f'{g} solvate -cp box.gro -cs spc216.gro -o solv.gro -p topol.top', cwd=d, tail=3)
    sh(f'{g} grompp -f {WORK}/em.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 2', cwd=d, tail=2)
    sh(f'{g} genion -s ions.tpr -o ionized.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15', inp='SOL\n', cwd=d, tail=2)
    sh(f'{g} grompp -f {WORK}/em.mdp -c ionized.gro -p topol.top -o em.tpr', cwd=d, tail=2)
    print('atoms', open(d / 'ionized.gro').readlines()[1].strip(), 'box', open(d / 'ionized.gro').readlines()[-1].strip()[:40])

d = WORK / 'systems/wt'
n = sh('nproc', quiet=True).strip()
GPUFLAGS = '-nb gpu -pme gpu -bonded gpu -update gpu -ntmpi 1'
sh(f'{GMX} mdrun -deffnm em -nb gpu -ntmpi 1', cwd=d, tail=6)
sh(f'{GMX} grompp -f {WORK}/nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 1', cwd=d, tail=2)
sh(f'{GMX} mdrun -deffnm nvt {GPUFLAGS} -ntomp {n}', cwd=d, tail=4)
sh(f'{GMX} grompp -f {WORK}/npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 1', cwd=d, tail=2)
sh(f'{GMX} mdrun -deffnm npt {GPUFLAGS} -ntomp {n}', cwd=d, tail=4)
sh(f'{GMX} grompp -f {WORK}/md.mdp -c npt.gro -t npt.cpt -p topol.top -o bench.tpr -maxwarn 1', cwd=d, tail=2)
one = sh(f'{GMX} mdrun -s bench.tpr -deffnm b1 {GPUFLAGS} -ntomp {n} -nsteps 30000 -resethway -noconfout', cwd=d, tail=3)
ngpu = int(sh('nvidia-smi -L | wc -l', quiet=True).strip())
res = {'one_run_all_cpu': re.findall(r'Performance:\s+([\d.]+)', one), 'ngpu': ngpu}
if ngpu > 1:
    half = int(n) // 2
    p = sh(f'({GMX} mdrun -s bench.tpr -deffnm b2 {GPUFLAGS} -ntomp {half} -gpu_id 0 -pin on -pinoffset 0 -nsteps 30000 -resethway -noconfout > b2.txt 2>&1 &'
           f' {GMX} mdrun -s bench.tpr -deffnm b3 {GPUFLAGS} -ntomp {half} -gpu_id 1 -pin on -pinoffset {half} -nsteps 30000 -resethway -noconfout > b3.txt 2>&1 & wait); grep -h Performance b2.txt b3.txt', cwd=d, check=False, tail=4)
    res['two_runs'] = re.findall(r'Performance:\s+([\d.]+)', p)
print('BENCHMARK ns/day', res)
json.dump(res, open(WORK / 'benchmark.json', 'w'))
for f in list(d.glob('b?.*')) + list(d.glob('*.trr')) + list(d.glob('#*')):
    f.unlink(missing_ok=True)
print('KERNEL H DONE')
